# DeepTest - Comprehensive Integration Testing

This notebook provides comprehensive integration testing for the S-57 conversion pipeline across all backends (PostGIS, GeoPackage, SpatiaLite).

#### Purpose

- **Full Pipeline Testing**: Validate S57Base, S57Advanced, and update mechanisms
- **Multi-Backend Validation**: Ensure consistency across PostGIS, GeoPackage, and SpatiaLite
- **Data Integrity**: Verify feature counts, geometry validity, and DSID stamping
- **Performance Baseline**: Track conversion times for regression testing

#### Workflow Overview

1. **Configuration** - Set test level (1-3), backend selection, and cleanup options
2. **Setup** - Validate data paths and database connections
3. **Test Execution** - Run full conversion pipeline across all backends
4. **Verification** - Inspect generated databases for data quality
5. **Reports** - Generate JSON, CSV, and TXT test results

#### Data Flow

```
ENC Files → S57Base/S57Advanced Conversion → All Backends → Comparison → Reports
```

#### Expected Outputs

- **Test Reports**: JSON, CSV, and TXT files with detailed results
- **Converted Data**: Test databases in all backends (if cleanup disabled)
- **Comparison Metrics**: Cross-backend consistency validation

#### Required Data

This notebook requires:
1. **ENC Test Data**: S-57 files in `data/ENC_ROOT/` directory
2. **PostgreSQL** (optional): Database with PostGIS extension for PostGIS backend
3. **Disk Space**: 50-100GB for full test with all backends (Level 1)

**Setup Instructions:** See `docs/getting-started/setup.md`
**Troubleshooting:** See `docs/reference/troubleshooting.md`

## 1. Configuration

  This notebook performs comprehensive integration testing of the entire S-57 conversion pipeline across all backends (PostGIS, GeoPackage, SpatiaLite). Configure your test level (1-3 for validation depth), select which backends to test, choose cleanup behavior, and optionally enable update mechanism testing. These settings control test scope, resource usage, and whether test artifacts are preserved after completion. **Note**: This is a diagnostic notebook, not a production conversion workflow—use `import_s57.ipynb` for regular ENC conversions.

In [ ]:
# =============================================================================
# NOTEBOOK CONFIGURATION - Adjust these parameters before running
# =============================================================================
# This notebook ONLY runs DeepTest - it does NOT perform regular conversions.
# For full conversion workflows, use import_s57.ipynb instead.

# --- DeepTest Configuration ---
# Core settings for the comprehensive test suite
db_schema = 's57_deeptest'  # PostGIS schema name for testing

deeptest_config = {
      # Test Depth (RECOMMENDED: 1 for first run)
      # 1 = Feature counts only (fastest, ~15 min)
      # 2 = + column schema validation (~1 hour)
      # 3 = + property completeness analysis (slowest, ~3-6 hours)
      'test_level': 3,

      # Backend Selection: Test PostGIS in addition to file-based backends
      # False = Test all three backends (requires .env with DB credentials)
      # True  = Test GeoPackage + SpatiaLite only (no DB required)
      # Note: GeoPackage and SpatiaLite are always tested if skip_postgis=True
      'skip_postgis': False,

      # Update Testing: Test incremental update mechanism (S57Updater)
      # False = Test updates (requires update data in data/ENC_ROOT_UPDATE)
      # True  = Skip update tests
      # Note: Automatically set to True if no update data is available
      'skip_updates': True,

      # Cleanup Behavior: What to do with test artifacts after successful completion
      # True  = Delete test outputs (saves disk space, faster re-runs)
      # False = Keep outputs for manual inspection and debugging
      'cleanup_on_success': False,

      # Pre-test Cleanup: Whether to clean existing outputs before starting
      # True  = Remove previous test results (guarantees clean slate)
      # False = Preserve existing results (incremental testing)
      'clean_output': True,

      # Column Filtering: Exclude these columns from cross-backend comparison
      # Geometry columns handled differently by each backend
      # (PostGIS: geometry, GeoPackage: geometry, SpatiaLite: wkb_geometry)
      'exclude_geometry_cols': ["geometry", "geom", "wkb_geometry"]
}


# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================
print("=" * 80)
print("✓ DeepTest Configuration Loaded Successfully!")
print("=" * 80)

# Test level with explanation
test_level_descriptions = {
  1: "High-level (feature counts only, fastest)",
  2: "Moderate (adds column schema validation)",
  3: "Comprehensive (adds property completeness analysis, slowest)"
}
print(f"\n📊 Test Level: {deeptest_config['test_level']} - {test_level_descriptions[deeptest_config['test_level']]}")

# Backend selection
if deeptest_config['skip_postgis']:
  print(f"\n🗃️  Backends: GeoPackage + SpatiaLite (PostGIS skipped)")
else:
  print(f"\n🗃️  Backends: PostGIS + GeoPackage + SpatiaLite (requires .env)")

# Update testing status
status = "DISABLED" if deeptest_config['skip_updates'] else "ENABLED"
print(f"🔄 Update Testing: {status}")

# Cleanup behavior
cleanup_desc = "DELETE after success" if deeptest_config['cleanup_on_success'] else "KEEP for inspection"
print(f"🧹 Test Artifacts: {cleanup_desc}")

print("\n" + "=" * 80)
print("⚠️  Paths and database parameters will be validated in STEP 2.1")
print("=" * 80)

### 1.1 Resource Requirements Quick Reference

**Disk Space**: Varies by test level and backend count
**Duration**: Depends on test level (1=fastest, 3=most thorough)
**Cleanup**: Use `cleanup_on_success=True` to reduce disk usage after completion

**See APPENDIX A.1 for test level comparison.**
**For detailed benchmarks and hardware requirements, see `docs/reference/technical-specs.md`**

### 1.2 Imports & Environment Setup

In [ ]:
# =============================================================================
# IMPORTS - Core dependencies for notebook
# =============================================================================

# --- Standard Library ---
import sys
import os
import logging
from pathlib import Path

# --- Environment Setup ---
from dotenv import load_dotenv

# --- Geospatial Libraries ---
from osgeo import gdal
import geopandas as gpd
import fiona

# --- Jupyter/Display ---
from IPython.display import display

# --- Fix PROJ_LIB Path (Common Conda/Jupyter Issue) ---
# Ensure GDAL/PROJ can find the coordinate database
conda_prefix = sys.prefix
possible_proj_lib = os.path.join(conda_prefix, 'share', 'proj')
if os.path.exists(possible_proj_lib):
    os.environ['PROJ_LIB'] = possible_proj_lib

# --- Environment Initialization ---
# Project root detection for test imports from uninstalled package
project_root = Path.cwd().parent.parent

# Add to path: Allows 'from tests.core__real_data...' to work
# Tests are in the repository but not installed as a package
sys.path.insert(0, str(project_root))

# --- Import DeepTest Classes ---
# These are only imported when running this notebook to avoid test dependencies
# in regular import workflows
from tests.core__real_data.deep_test_s57_workflow import TestConfig, S57DeepTester

# Load environment variables from .env file (at project root)
# Used for PostGIS connection: DB_NAME, DB_USER, DB_PASSWORD, DB_HOST, DB_PORT
load_dotenv(project_root / ".env")

# =============================================================================
# IMPORT VERIFICATION
# =============================================================================
print("=" * 80)
print("✓ All Core Imports Loaded Successfully!")
print("=" * 80)
print(f"\n📁 Project root: {project_root}")
print(f"✓ Python path includes project root (enables test imports)")
print(f"\n📦 Library Versions:")
print(f"   - GDAL: {gdal.__version__}")
print(f"   - GeoPandas: {gpd.__version__}")
print(f"   - Fiona: {fiona.__version__}")

# Warn if PostGIS will be tested but no .env file exists
if not deeptest_config['skip_postgis']:
  env_file = project_root / ".env"
  if env_file.exists():
      print(f"\n✓ .env file found (PostGIS credentials available)")
  else:
      print(f"\n⚠️  WARNING: .env file not found at {env_file}")
      print(f"   PostGIS tests will fail without database credentials.")
      print(f"   Either:")
      print(f"     1. Create .env file with DB credentials, or")
      print(f"     2. Set skip_postgis=True in deeptest_config")

print("\n" + "=" * 80)

### 1.3 Workflow Context

  **Purpose**: Comprehensive integration testing for S-57 conversion pipeline across all backends and conversion methods

  **When to Use This Notebook**:
  - **Before production release**: Validate conversion pipeline works correctly across all backends
  - **Regression testing**: Detect breaking changes in conversion logic or updates
  - **Certification requirements**: Prove data integrity and consistency (Test Level 3)
  - **Debugging conversions**: Identify which backend/method fails when issues occur
  - **Performance benchmarking**: Track conversion times and memory usage trends

  **What Gets Tested**:
  - **S57Base conversions**: One-to-one ENC to backend conversions (by_enc mode)
  - **S57Advanced conversions**: Layer-centric unified conversions with feature stamping (by_layer mode)
  - **Update mechanisms**: S57Updater incremental update testing (if data available)
  - **Multi-backend comparison**: Feature counts, schema validation, property completeness across PostGIS/GeoPackage/SpatiaLite
  - **Data integrity**: Geometry validity, DSID stamping, cross-backend consistency

  **Test Levels**:
  - **Level 1**: Feature count validation only (~15 minutes) - Quick CI/CD checks
  - **Level 2**: + Schema validation (~1 hour) - Development regression testing
  - **Level 3**: + Property completeness (~3-6 hours) - Production certification

  **Resource Requirements**:
  - **Disk Space**: 20-50GB depending on ENC count and test level (cleanup option available)
  - **Duration**: 15 minutes to 6 hours depending on test level and hardware
  - **Database**: PostGIS optional (can test file-based backends only)

  **Output**:
  - JSON/CSV/TXT test reports with detailed validation results
  - Test databases in all backends (if cleanup disabled)
  - Cross-backend consistency metrics

  **Next Steps**:
  - Review test reports in `test_output/` directory
  - Use findings to validate production deployment readiness
  - Address any failing validations before release
  - (Optional) Debug specific issues with preserved test databases

### 1.4 Environment Validation

Now that imports are complete, set up data paths and database connection parameters.

In [ ]:
# =============================================================================
# DATA PATHS VALIDATION AND DATABASE SETUP
# =============================================================================
# Validates required directories exist and configures database parameters for PostGIS.

# --- Data Source Paths Configuration ---
data_paths = {
  # ENC test data directory (used for S57Base and smaller tests)
  's57_data_dir': project_root / 'data' / 'ENC_ROOT',

  # ENC update files (used by S57Updater for incremental updates)
  's57_data_update_dir': project_root / 'data' / 'ENC_ROOT_UPDATE',

  # Test output directory (stores test reports and generated databases)
  'test_output_dir': project_root / 'test_output'
}

print("=" * 80)
print("🔍 Validating Data Paths and Database Configuration")
print("=" * 80)

# --- Validate primary ENC data directory (REQUIRED) ---
enc_data_dir = data_paths['s57_data_dir']
if enc_data_dir.exists():
  # Use rglob for recursive search: ENCs often organized in subdirectories by chart name
  enc_files = list(enc_data_dir.rglob("*.000"))
  total_size = sum(f.stat().st_size for f in enc_files)
  size_mb = total_size / (1024 * 1024)
  print(f"\n✅ Primary ENC Directory:")
  print(f"   Path: {enc_data_dir}")
  print(f"   ENC Files: {len(enc_files)} charts")
  print(f"   Total Size: {size_mb:.1f} MB")

  if len(enc_files) > 0:
    subdirs = sorted(set(f.parent.name for f in enc_files))
    if len(subdirs) > 0:
      print(f"   Organization: {len(subdirs)} subdirectory/ies")
      for subdir in subdirs:
        chart_files = [f for f in enc_files if f.parent.name == subdir]
        subdir_size = sum(f.stat().st_size for f in chart_files) / (1024 * 1024)
        print(f"      • {subdir}: {len(chart_files)} file(s), {subdir_size:.1f} MB")
else:
  print(f"\n❌ ERROR: Primary ENC directory not found!")
  print(f"   Expected: {enc_data_dir}")
  print(f"   Expected structure:")
  print(f"      data/ENC_ROOT/")
  print(f"      ├── US1GC09M/US1GC09M.000")
  print(f"      ├── US3CA52M/US3CA52M.000")
  print(f"      └── ... (more ENC files)")
  raise FileNotFoundError(f"Required ENC data directory missing: {enc_data_dir}")

# --- Validate update data directory (OPTIONAL) ---
# Auto-disable updates if missing to allow file-based-only testing
update_dir = data_paths['s57_data_update_dir']
if update_dir.exists():
  update_files = list(update_dir.rglob("*.000"))
  update_size = sum(f.stat().st_size for f in update_files)
  update_size_mb = update_size / (1024 * 1024)
  print(f"\n✅ Update Directory:")
  print(f"   Path: {update_dir}")
  print(f"   Update Files: {len(update_files)} charts")
  print(f"   Total Size: {update_size_mb:.1f} MB")

  if len(update_files) > 0:
    subdirs = sorted(set(f.parent.name for f in update_files))
    if len(subdirs) > 0:
      print(f"   Organization: {len(subdirs)} subdirectory/ies")
      for subdir in subdirs:
        chart_files = [f for f in update_files if f.parent.name == subdir]
        subdir_size = sum(f.stat().st_size for f in chart_files) / (1024 * 1024)
        print(f"      • {subdir}: {len(chart_files)} file(s), {subdir_size:.1f} MB")
  else:
    # Empty directory: auto-disable updates
    print(f"\n⚠️  Update directory is empty → disabling update testing")
    deeptest_config['skip_updates'] = True
else:
  # Directory missing: auto-disable updates to allow testing without update data
  print(f"\n⚠️  Update Directory Not Found:")
  print(f"   Expected: {update_dir}")
  print(f"   → Automatically disabling update testing (skip_updates=True)")
  deeptest_config['skip_updates'] = True

# --- Validate test output directory (AUTO-CREATE if needed) ---
output_dir = data_paths['test_output_dir']
if output_dir.exists():
  print(f"\n✅ Test Output Directory:")
  print(f"   Path: {output_dir}")
else:
  print(f"\n📁 Test Output Directory (will be created):")
  print(f"   Path: {output_dir}")

# =============================================================================
# DATABASE PARAMETER CONFIGURATION (PostGIS only)
# =============================================================================
db_params_test = {
  'dbname': os.getenv('DB_NAME'),
  'user': os.getenv('DB_USER'),
  'password': os.getenv('DB_PASSWORD'),
  'host': os.getenv('DB_HOST'),
  'port': os.getenv('DB_PORT')
}

# Display PostGIS configuration status
if not deeptest_config['skip_postgis']:
  print(f"\n🗄️  PostGIS Configuration:")
  if db_params_test['dbname']:
      print(f"   Database: {db_params_test['dbname']}")
      print(f"   Host: {db_params_test['host']}:{db_params_test['port']}")
      print(f"   Schema: {db_schema}")
      print(f"   Status: ✅ CONFIGURED")
  else:
      print(f"\n⚠️  ERROR: Database credentials not loaded from .env!")
      print(f"   Create .env file at {project_root / '.env'} with:")
      print(f"      DB_NAME=<database_name>")
      print(f"      DB_USER=<username>")
      print(f"      DB_PASSWORD=<password>")
      print(f"      DB_HOST=localhost")
      print(f"      DB_PORT=5432")
      print(f"   OR set skip_postgis=True to test file-based backends only")
else:
  print(f"\n🗄️  PostGIS: SKIPPED (skip_postgis=True)")

print("\n" + "=" * 80)
print("✅ Complete - Ready for DeepTest Initialization")
print("=" * 80)

## 2. Initialize DeepTest Configuration

Create the TestConfig object that encapsulates all test parameters. This configuration object is passed to the S57DeepTester class.

In [ ]:
# =============================================================================
# INITIALIZE DEEPTEST CONFIGURATION
# =============================================================================
# Create the TestConfig dataclass with all test parameters.

print("=" * 80)
print("🔧 Creating DeepTest Configuration")
print("=" * 80)

# Create TestConfig object with all parameters
test_config = TestConfig(
    # Input data paths
    s57_data_root=data_paths['s57_data_dir'],
    s57_update_root=data_paths['s57_data_update_dir'],
    test_output_dir=data_paths['test_output_dir'],
    # Test behavior parameters
    test_level=deeptest_config['test_level'],
    skip_postgis=deeptest_config['skip_postgis'],
    skip_updates=deeptest_config['skip_updates'],
    cleanup_on_success=deeptest_config['cleanup_on_success'],
    clean_output=deeptest_config['clean_output'],
    # Database configuration (only used if skip_postgis=False)
    postgis_config=db_params_test,
    test_schema_name=db_schema,
    # Column filtering for comparison
    exclude_extra_cols=deeptest_config['exclude_geometry_cols']
)

# =============================================================================
# DISPLAY CONFIGURATION SUMMARY
# =============================================================================
print(f"\n📋 Configuration Summary:")
print("-" * 80)

# Data paths
print(f"\n📁 Data Paths:")
print(f"   ENC Data:      {test_config.s57_data_root}")
print(f"   ENC Updates:   {test_config.s57_update_root}")
print(f"   Test Output:   {test_config.test_output_dir}")

# Test behavior
print(f"\n⚙️  Test Behavior:")
test_level_names = {1: "High-level (counts only)", 2: "Standard (+schema)", 3: "Deep (+properties)"}
print(f"   Test Level:    {test_config.test_level} - {test_level_names[test_config.test_level]}")
print(f"   Clean Output:  {'Yes' if test_config.clean_output else 'No'}")
print(f"   Cleanup:       {'Delete artifacts' if test_config.cleanup_on_success else 'Keep artifacts'}")

# Backend selection
print(f"\n🗄️  Backend Testing:")
if test_config.skip_postgis:
  print(f"   PostGIS:       ❌ SKIPPED")
  print(f"   GeoPackage:    ✅ ENABLED")
  print(f"   SpatiaLite:    ✅ ENABLED")
else:
  print(f"   PostGIS:       ✅ ENABLED ({db_params_test['host']}:{db_params_test['port']}/{db_params_test['dbname']})")
  print(f"   GeoPackage:    ✅ ENABLED")
  print(f"   SpatiaLite:    ✅ ENABLED")

# Update testing
print(f"\n🔄 Update Testing: {'❌ DISABLED' if test_config.skip_updates else '✅ ENABLED'}")

print("\n" + "=" * 80)
print("✅ Configuration Complete - Ready to Initialize S57DeepTester")
print("=" * 80)

## 3. Initialize S57DeepTester

### 3.1 Analyze Update Readiness (Optional - Safe to Skip)

Create the S57DeepTester instance and analyze which ENC charts have updates available. This compares DSID (Data Set Identification) records between the base data and update directories.

**What this shows:**
- `is_newer=True`: Chart has a newer version in the update directory (will be tested)
- `is_newer=False`: Chart has same version (no update available)
- `NO_UPDATE`: Chart only exists in base directory (no updates)
- `UPDATE`: Chart has newer version available

In [ ]:
# =============================================================================
# INITIALIZE S57DEEPTESTER
# =============================================================================

print("=" * 80)
print("🚀 Initializing S57DeepTester")
print("=" * 80)

# Initialize the tester with our configuration
try:
  tester = S57DeepTester(test_config)
  print("\n✅ S57DeepTester initialized successfully!")
  
  # Quick summary of test setup
  backend_count = sum(1 for backend in ['postgis', 'geopackage', 'spatialite']
                      if not (backend == 'postgis' and test_config.skip_postgis))
  print(f"   Backends configured: {backend_count}")
  print(f"   Test level: {test_config.test_level} (1=fast, 3=thorough)")
  print(f"   Ready for test execution")
except Exception as e:
  print(f"\n❌ ERROR: Failed to initialize S57DeepTester!")
  print(f"\n🔍 Likely Cause: Database connection or configuration issue")
  print(f"\n✅ Solution:")
  print(f"   1. Verify .env file has correct database credentials")
  print(f"   2. Check PostGIS database is running and accessible")
  print(f"   3. Or set skip_postgis=True to test file-based backends only")
  print(f"\n📋 Technical Details: {e}")
  import traceback
  traceback.print_exc()
  raise

# =============================================================================
# STEP 4.1: ANALYZE UPDATE READINESS (Optional Section)
# =============================================================================
# Compare DSID records between base and update directories to show which
# charts have newer versions available. This is informational only.

if not test_config.skip_updates:
  print("\n" + "=" * 80)
  print("📊 Analyzing Update Readiness")
  print("=" * 80)
  print("\nComparing ENC chart versions between base and update directories...\n")

  try:
      compare_df = tester.analyze_update_readiness()

      # Display summary statistics
      if not compare_df.empty:
          newer_count = len(compare_df[compare_df['is_newer'] == True])
          same_count = len(compare_df[compare_df['is_newer'] == False])
          total_count = len(compare_df)

          print(f"📈 Update Readiness Summary:")
          print(f"   Total charts analyzed:    {total_count}")
          print(f"   Newer versions available: {newer_count}")
          print(f"   Same versions:            {same_count}")

          if newer_count > 0:
              print(f"\n✅ {newer_count} chart(s) have updates - update testing will proceed")
          else:
              print(f"\n⚠️  No newer updates found - update testing will validate version check")

      print("\n📋 Detailed Comparison Results:")
      display(compare_df)

  except Exception as e:
      print(f"\n⚠️  Update readiness analysis failed (non-critical)")
      print(f"   Details: {e}")
      print(f"   → Proceeding with main test execution")
else:
  print("\n⏭️  Update readiness analysis SKIPPED (skip_updates=True)")

print("\n" + "=" * 80)
print("✅ Complete - Ready for Comprehensive Test Execution")
print("=" * 80)

## 4. Run Comprehensive DeepTest

Execute the full DeepTest workflow across all configured backends. This is the main test execution phase that:

1. **S57Base Conversion**: Convert all ENCs to each backend (basic geometry + attributes)
2. **S57Advanced Conversion**: Re-convert with feature stamping (dsid_* columns)
3. **Update Mechanism**: Apply incremental updates to test S57Updater
4. **Multi-level Comparison**: Validate consistency across backends at specified test level
5. **Report Generation**: Create JSON, CSV, and TXT reports with detailed results

**Estimated Duration**:
- Test Level 1: ~15 minutes (feature counts only)
- Test Level 2: ~1 hour (adds column validation)
- Test Level 3: ~3-6 hours (adds property completeness analysis)

In [ ]:
# =============================================================================
# RUN COMPREHENSIVE DEEPTEST
# =============================================================================
# Execute the full DeepTest workflow across all configured backends.
# This is the main test execution phase.

import pandas as pd
from datetime import datetime

print("=" * 80)
print("🎯 5: Running Comprehensive DeepTest")
print("=" * 80)

# Display test execution plan
print(f"\n📋 Test Execution Plan:")
print(f"   • S57Base Conversions:     All configured backends")
print(f"   • S57Advanced Conversions: All configured backends + feature stamping")
print(f"   • Update Testing:          {'Yes' if not test_config.skip_updates else 'No (skipped)'}")
print(f"   • Comparison Level:        {test_config.test_level}")
print(f"   • Cleanup After Success:   {'Yes' if test_config.cleanup_on_success else 'No (keep artifacts)'}")

# Duration estimates based on test level
duration_estimates = {
  1: "~15 minutes",
  2: "~1 hour",
  3: "~3-6 hours"
}
est_duration = duration_estimates.get(test_config.test_level, 'Unknown')
print(f"\n⏱️  Estimated Duration: {est_duration}")
print(f"\n⚠️  This will take a while... Go grab a coffee! ☕")
print(f"   Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

# Execute the comprehensive test
try:
  print("\n📦 Executing test phases...")
  print(f"   [1/4] S57Base conversions")
  print(f"   [2/4] S57Advanced conversions (with feature stamping)")
  if not test_config.skip_updates:
      print(f"   [3/4] Update mechanism testing")
  print(f"   [4/4] Backend comparison and verification")
  print()
  
  report = tester.run_comprehensive_test()

  # Success output
  print("\n" + "=" * 80)
  print("🎉 DEEPTEST COMPLETED SUCCESSFULLY!")
  print("=" * 80)
  print(f"   Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

  # Display report summary
  if report:
      print(f"\n📊 Test Results Summary:")
      print(f"   Output Directory: {test_config.test_output_dir}")

      # List generated report files
      if test_config.test_output_dir.exists():
          report_files = list(test_config.test_output_dir.glob("deeptest_report_*"))
          if report_files:
              print(f"   Reports Generated: {len(report_files)} file(s)")
              for report_file in sorted(report_files):
                  size_kb = report_file.stat().st_size / 1024
                  print(f"      • {report_file.name} ({size_kb:.1f}KB)")

      # Test completion summary
      print(f"\n✅ Conversion Completions:")
      print(f"   • S57Base conversions: Successfully completed")
      print(f"   • S57Advanced conversions: Feature stamping validated")
      if not test_config.skip_updates:
          print(f"   • Update mechanisms: Tested and verified")

      if test_config.cleanup_on_success:
          print(f"\n🧹 Cleanup: Test artifacts deleted (cleanup_on_success=True)")
      else:
          print(f"\n📁 Artifacts Preserved:")
          print(f"   Location: {test_config.test_output_dir}")
          print(f"   Review the reports above for detailed test results")

  print("\n" + "=" * 80)
  print("Next: Run Step 6 to verify and explore the test output")
  print("=" * 80)

except Exception as e:
  # Error handling with common failure scenarios
  print("\n" + "=" * 80)
  print("❌ DEEPTEST EXECUTION FAILED!")
  print("=" * 80)
  print(f"   Failed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
  
  error_type = type(e).__name__
  error_msg = str(e)
  
  print(f"\n🔍 Error Details:")
  print(f"   Type: {error_type}")
  print(f"   Message: {error_msg[:300]}")
  
  # Identify common failure scenarios and provide targeted solutions
  print(f"\n✅ Troubleshooting Guide:")
  print()
  
  if 'connection' in error_msg.lower() or 'postg' in error_msg.lower():
      print(f"   🔴 DATABASE CONNECTION ISSUE")
      print(f"      Your PostGIS database connection failed.")
      print(f"")
      print(f"      Recovery Steps:")
      print(f"      1. Verify PostGIS database is running:")
      print(f"         $ pg_isready -h {db_params_test['host']} -p {db_params_test['port']}")
      print(f"      2. Check credentials in .env file:")
      print(f"         $ cat {project_root / '.env'} | grep DB_")
      print(f"      3. Test direct connection:")
      print(f"         $ psql -h {db_params_test['host']} -U {db_params_test['user']} -d {db_params_test['dbname']}")
      print(f"      4. Or skip PostGIS and test file-based only:")
      print(f"         Set deeptest_config['skip_postgis'] = True")
      print(f"         Then re-run this cell")
      
  elif 'permission' in error_msg.lower() or 'access' in error_msg.lower():
      print(f"   🔴 FILE PERMISSION OR DISK SPACE ISSUE")
      print(f"      Cannot write to test output directory or insufficient disk space.")
      print(f"")
      print(f"      Recovery Steps:")
      print(f"      1. Check disk space:")
      print(f"         $ df -h {test_config.test_output_dir}")
      print(f"      2. Check directory permissions:")
      print(f"         $ ls -ld {test_config.test_output_dir}")
      print(f"      3. Make directory writable:")
      print(f"         $ chmod 755 {test_config.test_output_dir}")
      print(f"      4. Free up disk space (need 10-50GB depending on test level)")
      print(f"      5. Re-run this cell")
      
  elif 'gdal' in error_msg.lower() or 's57' in error_msg.lower():
      print(f"   🔴 GDAL/S-57 CONVERSION ISSUE")
      print(f"      ENC files could not be converted - may be corrupted or invalid.")
      print(f"")
      print(f"      Recovery Steps:")
      print(f"      1. Verify ENC files exist and are valid:")
      print(f"         $ find {test_config.s57_data_root} -name '*.000' -size +100k | head -5")
      print(f"      2. Check file integrity (should be > 100KB):")
      print(f"         $ ls -lh {test_config.s57_data_root}/*/*.000 | head -5")
      print(f"      3. Verify GDAL can read S-57 files:")
      print(f"         $ gdalinfo {test_config.s57_data_root}/*/*000 | head -20")
      print(f"      4. Try with smaller test level:")
      print(f"         Set deeptest_config['test_level'] = 1")
      print(f"         Then re-run this cell")
  else:
      print(f"   🔴 UNKNOWN FAILURE")
      print(f"")
      print(f"      Recovery Steps:")
      print(f"      1. Verify all prerequisites (Step 2.1):")
      print(f"         • ENC data exists: {test_config.s57_data_root}")
      print(f"         • Output directory writable: {test_config.test_output_dir}")
      print(f"         • Disk space available: df -h {test_config.test_output_dir}")
      print(f"      2. Try reducing test scope:")
      print(f"         Set deeptest_config['skip_postgis'] = True")
      print(f"         Set deeptest_config['test_level'] = 1")
      print(f"      3. Re-run this cell")
      print(f"      4. If still failing, review the full traceback below")
  
  print(f"\n📋 Full Error Trace:")
  print("-" * 80)
  import traceback
  traceback.print_exc()
  
  print("\n" + "=" * 80)
  print("⚠️  Do not proceed to Step 6 until this error is resolved")
  print("=" * 80)

## 5. Post-Test Verification and Data Exploration

This optional cell allows you to inspect the DeepTest output database to verify the conversion pipeline worked correctly.

**What this cell does:**
1. Opens the test-generated spatial database (GeoPackage or SpatiaLite)
2. Lists all S-57 layers that were successfully converted
3. Reads and inspects a sample layer (landmarks) to verify:
 - Data accessibility and integrity
 - Feature count and schema correctness
 - Presence of feature stamping columns (dsid_*) proving S57Advanced worked

**Note**: This serves as both verification that the conversion pipeline succeeded AND as a demonstration of how to programmatically access converted data.

In [ ]:
# =============================================================================
# POST-DEEPTEST VERIFICATION AND DATA EXPLORATION
# =============================================================================
# Opens test-generated database and inspects converted data to verify
# the S-57 conversion pipeline succeeded.

print("=" * 80)
print("🔍 Post-Test Verification and Data Exploration")
print("=" * 80)

# =============================================================================
# SECTION 1: TEST OUTPUT PATHS
# =============================================================================
test_output_path = data_paths['test_output_dir']

# Select database format to inspect (both available)
test_file = test_output_path / "s57_deeptest.sqlite"       # SpatiaLite format
# test_file = test_output_path / "s57_deeptest.gpkg"       # GeoPackage format

print(f"\n📁 Test Output Location:")
print(f"   Directory: {test_output_path}")
print(f"   Database:  {test_file.name}")

# Verify test artifacts exist
if not test_file.exists():
    print(f"\n⚠️  Test database not found: {test_file}")
    print(f"\n✅ Solution:")
    print(f"   1. Verify Step 5 (DeepTest Execution) completed successfully")
    print(f"   2. Check cleanup_on_success=False (artifacts may have been deleted)")
    print(f"   3. Run Step 5 again if needed")
    raise FileNotFoundError(f"Test artifacts not found. Run Step 5 first.")

# =============================================================================
# SECTION 2: DISCOVER ALL AVAILABLE LAYERS
# =============================================================================
print(f"\n🔍 Discovering S-57 layers in test database...")

try:
  # List all S-57 layers in the spatial database
  layer_names = fiona.listlayers(test_file)

  print(f"\n✅ Successfully opened database!")
  print(f"\n📊 Found {len(layer_names)} S-57 object classes:")

  # Display first 20 layers
  for i, name in enumerate(sorted(layer_names)[:20], 1):
      print(f"   {i:2d}. {name}")

  if len(layer_names) > 20:
      print(f"   ... and {len(layer_names) - 20} more")

  # Interpretation: Layer count indicates conversion completeness
  # Typical successful conversion has 60-100 layers representing all S-57 object classes
  print(f"\n   ℹ️  Interpretation:")
  if len(layer_names) >= 60:
      print(f"      ✅ High layer count ({len(layer_names)}) = Full S-57 dataset converted")
  elif len(layer_names) >= 20:
      print(f"      ⚠️  Moderate layer count ({len(layer_names)}) = Partial S-57 dataset")
  else:
      print(f"      ❌ Low layer count ({len(layer_names)}) = Possible conversion issue")

except fiona.errors.DriverError as e:
  # Database access error
  print(f"\n❌ ERROR: Could not open the database file!")
  print(f"\n✅ Solution:")
  print(f"   1. Verify DeepTest completed successfully (Step 5)")
  print(f"   2. Ensure file is not locked by another process")
  print(f"   3. Check file permissions: {test_file}")
  print(f"\n📋 Technical Details: {e}")
  raise

# =============================================================================
# SECTION 3: INSPECT A SAMPLE LAYER
# =============================================================================
print(f"\n" + "=" * 80)
print(f"🔍 Inspecting Sample Layer: LNDMRK (Landmarks)")
print("=" * 80)

selected_layer_name = 'lndmrk'  # Landmarks: lighthouses, towers, beacons, etc.

if selected_layer_name in layer_names:
  print(f"\n✅ Layer '{selected_layer_name}' found - loading data...")

  gdf = gpd.read_file(test_file, layer=selected_layer_name, engine="fiona")

  print(f"\n📊 Layer Statistics:")
  print(f"   Total Features: {len(gdf):,}")
  print(f"   Total Columns:  {len(gdf.columns)}")
  print(f"   Geometry Types: {gdf.geometry.geom_type.unique().tolist()}")

  # Validate feature stamping (proves S57Advanced conversion succeeded)
  stamping_cols = ['dsid_dsnm', 'dsid_edtn', 'dsid_updn']
  if all(col in gdf.columns for col in stamping_cols):
      print(f"\n✅ Feature Stamping Present (S57Advanced succeeded):")
      print(f"   Sample from first 3 features:")
      print(gdf[stamping_cols].head(3).to_string(index=False))
  else:
      print(f"\n⚠️  Feature stamping columns not found - check conversion")

  # =============================================================================
  # SECTION 4: DATA QUALITY VALIDATION
  # =============================================================================
  print(f"\n" + "-" * 80)
  print(f"📋 Schema Validation:")
  print("-" * 80)
  
  # Validate critical schema elements
  validation_checks = {
      'Geometry column': 'geometry' in gdf.columns,
      'Feature stamping': all(col in gdf.columns for col in stamping_cols),
      'S-57 schema (objl)': 'objl' in gdf.columns,
      'Geometry integrity': gdf.geometry.notna().all()
  }

  print(f"\n✅ Validation Results:")
  all_passed = True
  for check_name, passed in validation_checks.items():
      status = "✅" if passed else "❌"
      print(f"   {status} {check_name}")
      if not passed:
          all_passed = False

  if all_passed:
      print(f"\n🎉 All checks passed - S-57 conversion pipeline is working!")
  else:
      print(f"\n⚠️  Some checks failed - review conversion output")

else:
  # Layer not found
  print(f"\n⚠️  Layer '{selected_layer_name}' not found in database")
  print(f"\n💡 Try another layer:")
  for i, name in enumerate(sorted(layer_names)[:10], 1):
      print(f"   {i}. {name}")
  print(f"\n   Change selected_layer_name = '{sorted(layer_names)[0]}' and re-run")

print("\n" + "=" * 80)
print("✅ Complete - Verification Finished")
print("=" * 80)

## APPENDIX: Detailed Documentation

### A.1 Disk Space and Runtime Requirements

DeepTest creates multiple copies of converted data across all three backends for comparison. Plan for significantly more storage than a single conversion.

#### Disk Space by Test Level

| Level | Scope | S57Base | S57Advanced | Total | Notes |
|-------|-------|---------|------------|-------|-------|
| **1** | Feature counts only | ~10GB | ~10GB | **~20GB** | Varies by ENC count and backend |
| **2** | + Schema validation | ~10GB | ~10GB | **~20GB** | Same data, different validation |
| **3** | + Property analysis | ~10GB | ~10GB | **~20GB** | Same data, more thorough checking |

*Note: Storage usage is the same for all levels (all backends converted). Test level only affects comparison depth, not data storage.*

#### Per-Backend Storage

| Backend | Size (47 ENCs) | Size (6000 ENCs) | Notes |
|---------|---|---|-------|
| **S57Base (by_enc)** | 10GB (multiple databases) | 100GB | One database per ENC |
| **S57Advanced (by_layer)** | 10GB (single unified) | 100GB | Unified by object class |
| **Total (3 backends)** | ~50GB | ~600GB | Sum of all three |

#### Cleanup Behavior Impact

| Cleanup Setting | Disk Usage After Success | Use Case |
|---|---|---|
| **cleanup_on_success=True** | Minimal (~2GB reports) | CI/CD, automated testing, space-constrained |
| **cleanup_on_success=False** | 50-100GB (all backends preserved) | Debugging, manual inspection, analysis |

#### Expected Runtime

| Level | What Gets Tested | Duration |
|-------|---|---|
| **1** | Feature counts per layer | **~15 minutes** |
| **2** | Feature counts + column schema | **~1 hour** |
| **3** | Counts + schema + property completeness | **~3-6 hours** |

**Note**: All test levels perform full S57Base and S57Advanced conversions (the slow part). The test level only affects comparison verification time.

#### Performance Factors

DeepTest runtime depends on:

1. **ENC Count** (primary factor): Each ENC takes 10-30 seconds per backend
2. **Test Level** (verification only): Level 1 vs Level 3 adds significant comparison time
3. **Backend Order** (PostGIS first if enabled): PostGIS typically slower
4. **Hardware**: CPU cores, RAM, I/O (SSD vs HDD)

For more specific benchmarks and hardware recommendations, see `docs/reference/technical-specs.md`

### A.2 Testing Notebook Standards Exceptions

This is a testing/validation notebook with relaxed requirements compared to workflow notebooks:

**Acceptable Exceptions:**
- ✅ **No Workflow Context**: Testing notebook, not part of production pipeline
- ✅ **No BenchmarkLogger**: Uses test-specific reporting instead
- ✅ **Shorter title cell**: 38 lines acceptable (vs 40-50 for production)
- ✅ **Large reference sections**: Performance benchmarks needed for user planning

**Why These Exceptions Apply:**
- Testing notebooks serve diagnostic purposes, not production workflows
- Resource planning information is critical for users running the tests
- Test-specific reporting provides different value than BenchmarkLogger

### A.3 Test Level Detailed Comparison

| Test Level | Validation Type | Depth | Speed | Best For |
|---|---|---|---|---|
| **1** | Feature count validation | High-level only | ~15 min | Quick CI/CD checks, initial validation |
| **2** | + Column schema validation | Moderate | ~1 hour | Regression testing, development |
| **3** | + Property completeness analysis | Maximum | ~3-6 hours | Production release, certification |

#### When to Use Each Level

**Level 1 - Use if:**
- Quick CI/CD validation needed
- Just converted new ENC data
- Testing changes to conversion logic
- Space or time constrained

**Level 2 - Use if:**
- Regular regression testing
- Validating schema consistency
- Benchmarking performance trends
- ~1 hour acceptable wait time

**Level 3 - Use if:**
- Certification/production release
- Comprehensive data quality validation
- Detailed property completeness required
- Can wait 3-6 hours

### A.4 Backend-Specific Notes

#### PostGIS Backend
- **Status**: ✅ Fully recommended for DeepTest
- **Advantages**: Handles concurrent access safely, ACID transactions, best performance
- **Prerequisites**: PostgreSQL database with PostGIS extension, .env file with credentials
- **Notes**: All update mechanisms work reliably with PostGIS

#### GeoPackage Backend
- **Status**: ⚠️ Works but use with care
- **Advantages**: Portable, single-file format, works offline
- **Limitations**: Can cause corruption with concurrent access during updates
- **Recommendation**: Use for testing file-based conversions only; PostGIS for updates

#### SpatiaLite Backend
- **Status**: ⚠️ Works but use with care
- **Advantages**: Lightweight, embedded SQLite, no server needed
- **Limitations**: Can cause "database disk image is malformed" errors with concurrent access
- **Recommendation**: Use for testing file-based conversions only; PostGIS for updates

#### Optimization Tips

**To reduce disk usage:**
- Set `cleanup_on_success=True` to delete artifacts after completion
- Use `skip_postgis=True` to test file-based backends only (saves ~30%)
- Use smaller ENC subset if testing logic only

**To reduce runtime:**
- Use `test_level=1` for quick validation (~15 min vs 3-6 hours)
- Skip update testing: `skip_updates=True` (saves 20-30 minutes)
- Skip PostGIS: `skip_postgis=True` (saves ~30% of time)